### HW4: Classification of Green Fluorescent Protein

##### Name: Ananya Agrawal
##### Andrew ID: ananyaa2

##### 38615: Computational Modelling, Statistical Analysis and Machine Learning in Science - Homework 4


### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

### Reading the Training Set Features, Training Set Labels and Test Set Features

In [2]:
TRAIN_X = "Dataset/train_X.csv"
TRAIN_Y = "Dataset/train_y.csv"
KAGGLE_TEST_X = "Dataset/test_X.csv"

In [3]:
train_x = pd.read_csv(TRAIN_X, index_col=0)
train_y = pd.read_csv(TRAIN_Y, index_col=0)
kaggle_test_x = pd.read_csv(KAGGLE_TEST_X, index_col=0)

In [4]:
train_x

,ConstructedAASeq_cln,Id
0,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,11328
1,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,5781
2,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,13681
3,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,30804
4,SKGEELFTGVVPILVELDGDVNGHTFSVSGEGEGDATYGELTLKFI...,30813
...,...,...
33024,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,7024
33025,SKGEELFTGVVPTLVELDGDVNGHKFSVSGEGAGDATYSKLTLKFI...,14012
33026,SKGEELFTGVVPVLVELDGDVNGHKFSVSGEGEGDATYGKLTLKLI...,4140
33027,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,15193


In [5]:
train_y

,Brightness_Class,Id
0,0,11328
1,0,5781
2,0,13681
3,0,30804
4,0,30813
...,...,...
33024,0,7024
33025,0,14012
33026,0,4140
33027,1,15193


### Combining the feature set and label set mapping on common column - ID

In [6]:
train_data = pd.merge(train_x, train_y, on='Id')
train_data

,ConstructedAASeq_cln,Id,Brightness_Class
0,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,11328,0
1,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,5781,0
2,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,13681,0
3,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,30804,0
4,SKGEELFTGVVPILVELDGDVNGHTFSVSGEGEGDATYGELTLKFI...,30813,0
...,...,...,...
31024,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,7024,0
31025,SKGEELFTGVVPTLVELDGDVNGHKFSVSGEGAGDATYSKLTLKFI...,14012,0
31026,SKGEELFTGVVPVLVELDGDVNGHKFSVSGEGEGDATYGKLTLKLI...,4140,0
31027,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,15193,1


### Exploring the data

In [7]:
train_data.isnull().sum()

ConstructedAASeq_cln    0
Id                      0
Brightness_Class        0
dtype: int64

In [8]:
train_data['Brightness_Class'].value_counts()

Brightness_Class
0    18948
1    12081
Name: count, dtype: int64

In [9]:
train_data.describe()

,Id,Brightness_Class
count,31029.000000,31029.000000
mean,15514.000000,0.389345
std,8957.445088,0.487610
min,0.000000,0.000000
25%,7757.000000,0.000000
50%,15514.000000,0.000000
75%,23271.000000,1.000000
max,31028.000000,1.000000


In [10]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31029 entries, 0 to 31028
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   ConstructedAASeq_cln  31029 non-null  object
 1   Id                    31029 non-null  int64 
 2   Brightness_Class      31029 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 727.4+ KB


In [11]:
train_data.dtypes

ConstructedAASeq_cln    object
Id                       int64
Brightness_Class         int64
dtype: object

In [12]:
seq_len = train_data['ConstructedAASeq_cln']
len(seq_len.unique())

31029

## Exploring the features - Descriptors for amino acid sequences

### Feature Mapping 1 - DPPS

In [14]:
dpps = pd.read_csv('Dataset/descriptors/DPPS.csv', skiprows=[0,1])
dpps_mapping = {}

for index, row in dpps.iterrows():
    temp = []
    for i in range(1, 11):
        temp.append(row['D' + str(i)])
    dpps_mapping[row['AA_1']] = temp

def map_dpps(seq):
    res = []
    for aa in seq:
        res.append(dpps_mapping[aa])
    return np.array(res).flatten().tolist()

In [15]:
dpps_mapping

{'A': [-1.02, -2.88, -0.56, 0.36, -6.15, -1.68, 0.04, -2.51, -1.94, -0.01],
 'R': [1.99, 4.13, -4.41, -1.02, 4.78, 3.04, -9.06, 6.71, 4.41, 0.07],
 'N': [-2.19, 1.86, 0.38, -0.13, -2.3, 1.41, -5.71, -1.11, 1.73, -0.19],
 'D': [-6.6, 3.32, 1.61, 0.36, -3.25, 1.95, -7.36, 0.14, 1.24, -0.15],
 'C': [0.21, 1.12, 3.42, -0.68, -2.27, -1.22, 3.11, -2.98, -1.7, 1.57],
 'Q': [-0.47, 1.16, -0.57, 0.69, 0.39, 1.93, -5.46, -0.84, 1.93, 0.85],
 'E': [-5.39, 0.65, -0.98, 1.39, -0.23, 2.51, -6.84, -0.68, 1.41, 1.28],
 'G': [-2.86, -5.0, -2.97, 0.53, -11.45, 1.89, -2.11, -3.99, -2.16, -0.76],
 'H': [0.73, 2.68, -0.66, -1.89, 1.6, 1.13, -1.94, -0.11, 0.44, 0.15],
 'I': [1.91, -3.13, 0.01, 1.14, 2.7, -4.55, 8.93, 0.18, -1.1, -0.76],
 'L': [1.64, -2.57, 0.0, 1.35, 2.62, -2.65, 7.72, 0.05, -1.03, -1.81],
 'K': [2.47, 1.54, -4.28, -0.86, 2.77, 2.06, -6.18, 2.05, 2.19, -1.65],
 'M': [1.93, -0.01, 1.21, 0.99, 2.79, -0.56, 5.33, -0.87, -0.99, -1.09],
 'F': [2.68, 0.84, 2.22, 0.71, 5.02, -0.3, 8.6, 1.13, -1.4,

In [16]:
len(map_dpps("SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLSYGAQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYDYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"))

2370

In [18]:
dpps_columns = pd.DataFrame(train_data['ConstructedAASeq_cln'].apply(map_dpps).tolist(), columns=[f'feature_dpps_{i+1}' for i in range(2370)])

In [19]:
dpps_columns

,feature_dpps_1,feature_dpps_2,feature_dpps_3,feature_dpps_4,feature_dpps_5,feature_dpps_6,feature_dpps_7,feature_dpps_8,feature_dpps_9,feature_dpps_10,...,feature_dpps_2361,feature_dpps_2362,feature_dpps_2363,feature_dpps_2364,feature_dpps_2365,feature_dpps_2366,feature_dpps_2367,feature_dpps_2368,feature_dpps_2369,feature_dpps_2370
0,-1.76,-0.19,1.06,-0.69,-5.72,0.14,-4.14,-2.42,-0.13,0.69,...,2.47,1.54,-4.28,-0.86,2.77,2.06,-6.18,2.05,2.19,-1.65
1,-1.76,-0.19,1.06,-0.69,-5.72,0.14,-4.14,-2.42,-0.13,0.69,...,2.47,1.54,-4.28,-0.86,2.77,2.06,-6.18,2.05,2.19,-1.65
2,-1.76,-0.19,1.06,-0.69,-5.72,0.14,-4.14,-2.42,-0.13,0.69,...,2.47,1.54,-4.28,-0.86,2.77,2.06,-6.18,2.05,2.19,-1.65
3,-1.76,-0.19,1.06,-0.69,-5.72,0.14,-4.14,-2.42,-0.13,0.69,...,2.47,1.54,-4.28,-0.86,2.77,2.06,-6.18,2.05,2.19,-1.65
4,-1.76,-0.19,1.06,-0.69,-5.72,0.14,-4.14,-2.42,-0.13,0.69,...,2.47,1.54,-4.28,-0.86,2.77,2.06,-6.18,2.05,2.19,-1.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31024,-1.76,-0.19,1.06,-0.69,-5.72,0.14,-4.14,-2.42,-0.13,0.69,...,2.47,1.54,-4.28,-0.86,2.77,2.06,-6.18,2.05,2.19,-1.65
31025,-1.76,-0.19,1.06,-0.69,-5.72,0.14,-4.14,-2.42,-0.13,0.69,...,2.47,1.54,-4.28,-0.86,2.77,2.06,-6.18,2.05,2.19,-1.65
31026,-1.76,-0.19,1.06,-0.69,-5.72,0.14,-4.14,-2.42,-0.13,0.69,...,2.47,1.54,-4.28,-0.86,2.77,2.06,-6.18,2.05,2.19,-1.65
31027,-1.76,-0.19,1.06,-0.69,-5.72,0.14,-4.14,-2.42,-0.13,0.69,...,2.47,1.54,-4.28,-0.86,2.77,2.06,-6.18,2.05,2.19,-1.65


In [20]:
dpps_columns.to_csv('dpps.csv', index=None)

### Feature Mapping 2 - MS-WHIM

In [21]:
mswhim = pd.read_csv('Dataset/descriptors/MS-WHIM.csv', skiprows=[0,1])
mswhim_mapping = {}

for index, row in mswhim.iterrows():
    mswhim_mapping[row['AA_1']] = [row['Ist'], row['2nd'], row['3rd']]

def map_mswhim(seq):
    res = []
    for aa in seq:
        res.append(mswhim_mapping[aa])
    return np.array(res).flatten().tolist()

In [22]:
mswhim_mapping

{'A': [-0.73, 0.2, -0.62],
 'R': [-0.22, 0.27, 1.0],
 'N': [0.14, 0.2, -0.66],
 'D': [0.11, -1.0, -0.96],
 'C': [-0.66, 0.26, -0.27],
 'Q': [0.3, 1.0, -0.3],
 'E': [0.24, -0.39, -0.04],
 'G': [-0.31, -0.28, -0.75],
 'H': [0.84, 0.67, -0.78],
 'I': [-0.91, 0.83, -0.25],
 'L': [-0.74, 0.72, -0.16],
 'K': [-0.51, 0.08, 0.6],
 'M': [-0.7, 1.0, -0.32],
 'F': [0.76, 0.85, -0.34],
 'P': [-0.43, 0.73, -0.6],
 'S': [-0.8, 0.61, -1.0],
 'T': [-0.58, 0.85, -0.89],
 'W': [1.0, 0.98, -0.47],
 'Y': [0.97, 0.66, -0.16],
 'V': [-1.0, 0.79, -0.58]}

In [23]:
len(map_mswhim("SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLSYGAQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYDYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"))

711

In [24]:
whim_columns = pd.DataFrame(train_data['ConstructedAASeq_cln'].apply(map_mswhim).tolist(), columns=[f'feature_ms_whim_{i+1}' for i in range(711)])

In [25]:
whim_columns

,feature_ms_whim_1,feature_ms_whim_2,feature_ms_whim_3,feature_ms_whim_4,feature_ms_whim_5,feature_ms_whim_6,feature_ms_whim_7,feature_ms_whim_8,feature_ms_whim_9,feature_ms_whim_10,...,feature_ms_whim_702,feature_ms_whim_703,feature_ms_whim_704,feature_ms_whim_705,feature_ms_whim_706,feature_ms_whim_707,feature_ms_whim_708,feature_ms_whim_709,feature_ms_whim_710,feature_ms_whim_711
0,-0.8,0.61,-1.0,-0.51,0.08,0.6,-0.31,-0.28,-0.75,0.24,...,-0.04,-0.74,0.72,-0.16,0.97,0.66,-0.16,-0.51,0.08,0.6
1,-0.8,0.61,-1.0,-0.51,0.08,0.6,-0.31,-0.28,-0.75,0.24,...,-0.04,-0.74,0.72,-0.16,0.97,0.66,-0.16,-0.51,0.08,0.6
2,-0.8,0.61,-1.0,-0.51,0.08,0.6,-0.31,-0.28,-0.75,0.24,...,-0.04,-0.74,0.72,-0.16,0.97,0.66,-0.16,-0.51,0.08,0.6
3,-0.8,0.61,-1.0,-0.51,0.08,0.6,-0.31,-0.28,-0.75,0.24,...,-0.04,-0.74,0.72,-0.16,0.97,0.66,-0.16,-0.51,0.08,0.6
4,-0.8,0.61,-1.0,-0.51,0.08,0.6,-0.31,-0.28,-0.75,0.24,...,-0.04,-0.74,0.72,-0.16,0.97,0.66,-0.16,-0.51,0.08,0.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31024,-0.8,0.61,-1.0,-0.51,0.08,0.6,-0.31,-0.28,-0.75,0.24,...,-0.04,0.30,1.00,-0.30,0.97,0.66,-0.16,-0.51,0.08,0.6
31025,-0.8,0.61,-1.0,-0.51,0.08,0.6,-0.31,-0.28,-0.75,0.24,...,-0.04,-0.74,0.72,-0.16,-0.66,0.26,-0.27,-0.51,0.08,0.6
31026,-0.8,0.61,-1.0,-0.51,0.08,0.6,-0.31,-0.28,-0.75,0.24,...,-0.04,-0.74,0.72,-0.16,0.97,0.66,-0.16,-0.51,0.08,0.6
31027,-0.8,0.61,-1.0,-0.51,0.08,0.6,-0.31,-0.28,-0.75,0.24,...,-0.04,0.30,1.00,-0.30,0.97,0.66,-0.16,-0.51,0.08,0.6


In [26]:
whim_columns.to_csv('ms-whim.csv', index=None)

### Feature Mapping 3 - Physical

In [27]:
physical = pd.read_csv('Dataset/descriptors/Physical.csv', skiprows=[0,1])
physical_mapping = {}
for index, row in physical.iterrows():
    physical_mapping[row['AA_1']] = [row['Vol'], row['Hydro']]

def map_physical(seq):
    res = []
    for aa in seq:
        res.append(physical_mapping[aa])
    return np.array(res).flatten().tolist()

In [28]:
physical_mapping

{'A': [-2.9, -1.03],
 'R': [2.41, 1.31],
 'N': [-0.68, 0.79],
 'D': [-0.92, 1.23],
 'C': [-1.89, 0.15],
 'Q': [0.36, 1.09],
 'E': [0.16, 1.28],
 'G': [-4.04, 0.01],
 'H': [0.83, 1.15],
 'I': [0.51, -1.32],
 'L': [0.52, -1.4],
 'K': [0.92, 1.23],
 'M': [0.92, -1.42],
 'F': [2.22, -1.47],
 'P': [-1.25, -0.64],
 'S': [-2.36, 0.38],
 'T': [-1.19, 0.28],
 'W': [4.28, -0.18],
 'Y': [2.75, -0.18],
 'V': [-0.65, -1.27]}

In [29]:
len(map_physical("SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLSYGAQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYDYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"))

474

In [30]:
phy_columns = pd.DataFrame(train_data['ConstructedAASeq_cln'].apply(map_physical).tolist(), columns=[f'feature_physical_{i+1}' for i in range(474)])

In [31]:
phy_columns

,feature_physical_1,feature_physical_2,feature_physical_3,feature_physical_4,feature_physical_5,feature_physical_6,feature_physical_7,feature_physical_8,feature_physical_9,feature_physical_10,...,feature_physical_465,feature_physical_466,feature_physical_467,feature_physical_468,feature_physical_469,feature_physical_470,feature_physical_471,feature_physical_472,feature_physical_473,feature_physical_474
0,-2.36,0.38,0.92,1.23,-4.04,0.01,0.16,1.28,0.16,1.28,...,-0.92,1.23,0.16,1.28,0.52,-1.40,2.75,-0.18,0.92,1.23
1,-2.36,0.38,0.92,1.23,-4.04,0.01,0.16,1.28,0.16,1.28,...,-0.92,1.23,0.16,1.28,0.52,-1.40,2.75,-0.18,0.92,1.23
2,-2.36,0.38,0.92,1.23,-4.04,0.01,0.16,1.28,0.16,1.28,...,-0.92,1.23,0.16,1.28,0.52,-1.40,2.75,-0.18,0.92,1.23
3,-2.36,0.38,0.92,1.23,-4.04,0.01,0.16,1.28,0.16,1.28,...,-0.92,1.23,0.16,1.28,0.52,-1.40,2.75,-0.18,0.92,1.23
4,-2.36,0.38,0.92,1.23,-4.04,0.01,0.16,1.28,0.16,1.28,...,-0.92,1.23,0.16,1.28,0.52,-1.40,2.75,-0.18,0.92,1.23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31024,-2.36,0.38,0.92,1.23,-4.04,0.01,0.16,1.28,0.16,1.28,...,-0.92,1.23,0.16,1.28,0.36,1.09,2.75,-0.18,0.92,1.23
31025,-2.36,0.38,0.92,1.23,-4.04,0.01,0.16,1.28,0.16,1.28,...,-0.92,1.23,0.16,1.28,0.52,-1.40,-1.89,0.15,0.92,1.23
31026,-2.36,0.38,0.92,1.23,-4.04,0.01,0.16,1.28,0.16,1.28,...,-0.92,1.23,0.16,1.28,0.52,-1.40,2.75,-0.18,0.92,1.23
31027,-2.36,0.38,0.92,1.23,-4.04,0.01,0.16,1.28,0.16,1.28,...,-0.92,1.23,0.16,1.28,0.36,1.09,2.75,-0.18,0.92,1.23


In [32]:
phy_columns.to_csv('physical.csv', index=None)

### Feature Mapping 4 - ST-Scale

In [33]:
stscale = pd.read_csv('Dataset/descriptors/ST-scale.csv', skiprows=[0,1])
stscale_mapping = {}
for index, row in stscale.iterrows():
    temp = []
    for i in range(1, 9):
        temp.append(row['ST' + str(i)])
    stscale_mapping[row['AA_1']] = temp

def map_stscale(seq):
    res = []
    for aa in seq:
        res.append(stscale_mapping[aa])
    return np.array(res).flatten().tolist()

In [34]:
stscale_mapping

{'A': [-1.552, -0.791, -0.627, 0.237, -0.461, -2.229, 0.283, 1.221],
 'R': [-0.059, 0.731, -0.013, -0.096, -0.253, 0.3, 1.256, 0.854],
 'N': [-0.888, -0.057, -0.651, -0.214, 0.917, 0.164, -0.14, -0.166],
 'D': [-0.907, -0.054, -0.781, -0.248, 1.12, 0.101, -0.245, -0.075],
 'C': [-1.276, -0.401, 0.134, 0.859, -0.196, -0.72, 0.639, -0.857],
 'Q': [-0.622, 0.228, -0.193, -0.105, 0.418, 0.474, 0.172, 0.408],
 'E': [-0.629, 0.39, -0.38, -0.366, 0.635, 0.514, 0.175, 0.367],
 'G': [-1.844, -0.018, -0.184, 0.573, -0.728, -3.317, 0.166, 2.522],
 'H': [-0.225, 0.361, 0.079, -1.037, 0.568, 0.273, 1.208, -0.001],
 'I': [-0.785, -1.01, -0.349, -0.097, -0.402, 1.091, -0.139, -0.764],
 'L': [-0.826, -0.379, 0.038, -0.059, -0.625, 1.025, -0.229, -0.129],
 'K': [-0.504, 0.245, 0.297, -0.065, -0.387, 1.011, 0.525, 0.553],
 'M': [-0.693, 0.498, 0.658, 0.457, -0.231, 1.064, 0.248, -0.778],
 'F': [-0.019, 0.024, 1.08, -0.22, -0.937, 0.57, -0.357, 0.278],
 'P': [-1.049, -0.407, -0.067, -0.066, -0.813, -0.89

In [35]:
len(map_stscale("SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLSYGAQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYDYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"))

1896

In [36]:
stscale_columns = pd.DataFrame(train_data['ConstructedAASeq_cln'].apply(map_stscale).tolist(), columns=[f'feature_st_scale_{i+1}' for i in range(1896)])

In [37]:
stscale_columns

,feature_st_scale_1,feature_st_scale_2,feature_st_scale_3,feature_st_scale_4,feature_st_scale_5,feature_st_scale_6,feature_st_scale_7,feature_st_scale_8,feature_st_scale_9,feature_st_scale_10,...,feature_st_scale_1887,feature_st_scale_1888,feature_st_scale_1889,feature_st_scale_1890,feature_st_scale_1891,feature_st_scale_1892,feature_st_scale_1893,feature_st_scale_1894,feature_st_scale_1895,feature_st_scale_1896
0,-1.343,-0.311,-0.917,-0.049,0.549,-1.533,0.166,0.28,-0.504,0.245,...,-1.099,0.162,-0.504,0.245,0.297,-0.065,-0.387,1.011,0.525,0.553
1,-1.343,-0.311,-0.917,-0.049,0.549,-1.533,0.166,0.28,-0.504,0.245,...,-1.099,0.162,-0.504,0.245,0.297,-0.065,-0.387,1.011,0.525,0.553
2,-1.343,-0.311,-0.917,-0.049,0.549,-1.533,0.166,0.28,-0.504,0.245,...,-1.099,0.162,-0.504,0.245,0.297,-0.065,-0.387,1.011,0.525,0.553
3,-1.343,-0.311,-0.917,-0.049,0.549,-1.533,0.166,0.28,-0.504,0.245,...,-1.099,0.162,-0.504,0.245,0.297,-0.065,-0.387,1.011,0.525,0.553
4,-1.343,-0.311,-0.917,-0.049,0.549,-1.533,0.166,0.28,-0.504,0.245,...,-1.099,0.162,-0.504,0.245,0.297,-0.065,-0.387,1.011,0.525,0.553
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31024,-1.343,-0.311,-0.917,-0.049,0.549,-1.533,0.166,0.28,-0.504,0.245,...,-1.099,0.162,-0.504,0.245,0.297,-0.065,-0.387,1.011,0.525,0.553
31025,-1.343,-0.311,-0.917,-0.049,0.549,-1.533,0.166,0.28,-0.504,0.245,...,0.639,-0.857,-0.504,0.245,0.297,-0.065,-0.387,1.011,0.525,0.553
31026,-1.343,-0.311,-0.917,-0.049,0.549,-1.533,0.166,0.28,-0.504,0.245,...,-1.099,0.162,-0.504,0.245,0.297,-0.065,-0.387,1.011,0.525,0.553
31027,-1.343,-0.311,-0.917,-0.049,0.549,-1.533,0.166,0.28,-0.504,0.245,...,-1.099,0.162,-0.504,0.245,0.297,-0.065,-0.387,1.011,0.525,0.553


In [38]:
stscale_columns.to_csv('st-scale.csv', index=None)

### Feature Mapping 5 - T-Scale

In [39]:
tscale = pd.read_csv('Dataset/descriptors/T-scale.csv', skiprows=[0,1])
tscale_mapping = {}
for index, row in tscale.iterrows():
    temp = []
    for i in range(1, 6):
        temp.append(row['T' + str(i)])
    tscale_mapping[row['AA_1']] = temp

def map_tscale(seq):
    res = []
    for aa in seq:
        res.append(tscale_mapping[aa])
    return np.array(res).flatten().tolist()

In [40]:
tscale_mapping

{'A': [-9.11, -1.63, 0.63, 1.04, 2.26],
 'R': [0.23, 3.89, -1.16, -0.39, -0.06],
 'N': [-4.62, 0.66, 1.16, -0.22, 0.93],
 'D': [-4.65, 0.75, 1.39, -0.4, 1.05],
 'C': [-7.35, -0.86, -0.33, 0.8, 0.98],
 'Q': [-3.0, 1.72, 0.28, -0.39, 0.33],
 'E': [-3.03, 1.82, 0.51, -0.58, 0.43],
 'G': [-10.61, -1.21, -0.12, 0.75, 3.25],
 'H': [-1.01, -1.31, 0.01, -1.81, -0.21],
 'I': [-4.25, -0.28, -0.15, 1.4, -0.21],
 'L': [-4.38, 0.28, -0.49, 1.45, 0.02],
 'K': [-2.59, 2.34, -1.69, 0.41, -0.21],
 'M': [-4.08, 0.98, -2.34, 1.64, -0.79],
 'F': [0.49, -0.94, -0.63, -1.27, -0.44],
 'P': [-5.11, -3.54, -0.53, -0.36, -0.29],
 'S': [-7.44, -0.65, 0.68, -0.17, 1.58],
 'T': [-5.97, -0.62, 1.11, 0.31, 0.95],
 'W': [5.73, -2.67, -0.07, -1.96, -0.54],
 'Y': [2.08, -0.47, 0.07, -1.67, -0.35],
 'V': [-5.87, -0.94, 0.28, 1.1, 0.48]}

In [41]:
len(map_tscale("SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLSYGAQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYDYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"))

1185

In [42]:
tscale_columns = pd.DataFrame(train_data['ConstructedAASeq_cln'].apply(map_tscale).tolist(), columns=[f'feature_t_scale_{i+1}' for i in range(1185)])

In [43]:
tscale_columns

,feature_t_scale_1,feature_t_scale_2,feature_t_scale_3,feature_t_scale_4,feature_t_scale_5,feature_t_scale_6,feature_t_scale_7,feature_t_scale_8,feature_t_scale_9,feature_t_scale_10,...,feature_t_scale_1176,feature_t_scale_1177,feature_t_scale_1178,feature_t_scale_1179,feature_t_scale_1180,feature_t_scale_1181,feature_t_scale_1182,feature_t_scale_1183,feature_t_scale_1184,feature_t_scale_1185
0,-7.44,-0.65,0.68,-0.17,1.58,-2.59,2.34,-1.69,0.41,-0.21,...,2.08,-0.47,0.07,-1.67,-0.35,-2.59,2.34,-1.69,0.41,-0.21
1,-7.44,-0.65,0.68,-0.17,1.58,-2.59,2.34,-1.69,0.41,-0.21,...,2.08,-0.47,0.07,-1.67,-0.35,-2.59,2.34,-1.69,0.41,-0.21
2,-7.44,-0.65,0.68,-0.17,1.58,-2.59,2.34,-1.69,0.41,-0.21,...,2.08,-0.47,0.07,-1.67,-0.35,-2.59,2.34,-1.69,0.41,-0.21
3,-7.44,-0.65,0.68,-0.17,1.58,-2.59,2.34,-1.69,0.41,-0.21,...,2.08,-0.47,0.07,-1.67,-0.35,-2.59,2.34,-1.69,0.41,-0.21
4,-7.44,-0.65,0.68,-0.17,1.58,-2.59,2.34,-1.69,0.41,-0.21,...,2.08,-0.47,0.07,-1.67,-0.35,-2.59,2.34,-1.69,0.41,-0.21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31024,-7.44,-0.65,0.68,-0.17,1.58,-2.59,2.34,-1.69,0.41,-0.21,...,2.08,-0.47,0.07,-1.67,-0.35,-2.59,2.34,-1.69,0.41,-0.21
31025,-7.44,-0.65,0.68,-0.17,1.58,-2.59,2.34,-1.69,0.41,-0.21,...,-7.35,-0.86,-0.33,0.80,0.98,-2.59,2.34,-1.69,0.41,-0.21
31026,-7.44,-0.65,0.68,-0.17,1.58,-2.59,2.34,-1.69,0.41,-0.21,...,2.08,-0.47,0.07,-1.67,-0.35,-2.59,2.34,-1.69,0.41,-0.21
31027,-7.44,-0.65,0.68,-0.17,1.58,-2.59,2.34,-1.69,0.41,-0.21,...,2.08,-0.47,0.07,-1.67,-0.35,-2.59,2.34,-1.69,0.41,-0.21


In [44]:
tscale_columns.to_csv('t-scale.csv', index=None)

### Feature Mapping 6 - VHSE-Scale

In [45]:
vhsescale = pd.read_csv('Dataset/descriptors/VHSE-scale.csv', skiprows=[0,1])
vhsescale_mapping = {}
for index, row in vhsescale.iterrows():
    temp = []
    for i in range(1, 9):
        temp.append(row['VHSE' + str(i)])
    vhsescale_mapping[row['AA_1']] = temp

def map_vhsescale(seq):
    res = []
    for aa in seq:
        res.append(vhsescale_mapping[aa])
    return np.array(res).flatten().tolist()

In [46]:
vhsescale_mapping

{'A': [0.15, -1.11, -1.35, -0.92, 0.02, -0.91, 0.36, -0.48],
 'R': [-1.47, 1.45, 1.24, 1.27, 1.55, 1.47, 1.3, 0.83],
 'N': [-0.99, 0.0, -0.37, 0.69, -0.55, 0.85, 0.73, -0.8],
 'D': [-1.15, 0.67, -0.41, -0.01, -2.68, 1.31, 0.03, 0.56],
 'C': [0.18, -1.67, -0.46, -0.21, 0.0, 1.2, -1.61, -0.19],
 'Q': [-0.96, 0.12, 0.18, 0.16, 0.09, 0.42, -0.2, -0.41],
 'E': [-1.18, 0.4, 0.1, 0.36, -2.16, -0.17, 0.91, 0.02],
 'G': [-0.2, -1.53, -2.63, 2.28, -0.53, -1.18, 2.01, -1.34],
 'H': [-0.43, -0.25, 0.37, 0.19, 0.51, 1.28, 0.93, 0.65],
 'I': [1.27, -0.14, 0.3, -1.8, 0.3, -1.61, -0.16, -0.13],
 'L': [1.36, 0.07, 0.26, -0.8, 0.22, -1.37, 0.08, -0.62],
 'K': [-1.17, 0.7, 0.7, 0.8, 1.64, 0.67, 1.63, 0.13],
 'M': [1.01, -0.53, 0.43, 0.0, 0.23, 0.1, -0.86, -0.68],
 'F': [1.52, 0.61, 0.96, -0.16, 0.25, 0.28, -1.33, -0.2],
 'P': [0.22, -0.17, -0.5, 0.05, -0.01, -1.34, -0.19, 3.56],
 'S': [-0.67, -0.86, -1.07, -0.41, -0.32, 0.27, -0.64, 0.11],
 'T': [-0.34, -0.51, -0.55, -1.06, -0.06, -0.01, -0.79, 0.39],
 '

In [47]:
len(map_vhsescale("SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLSYGAQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYDYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"))

1896

In [48]:
vhse_columns = pd.DataFrame(train_data['ConstructedAASeq_cln'].apply(map_vhsescale).tolist(), columns=[f'feature_vhse_scale_{i+1}' for i in range(1896)])

In [49]:
vhse_columns

,feature_vhse_scale_1,feature_vhse_scale_2,feature_vhse_scale_3,feature_vhse_scale_4,feature_vhse_scale_5,feature_vhse_scale_6,feature_vhse_scale_7,feature_vhse_scale_8,feature_vhse_scale_9,feature_vhse_scale_10,...,feature_vhse_scale_1887,feature_vhse_scale_1888,feature_vhse_scale_1889,feature_vhse_scale_1890,feature_vhse_scale_1891,feature_vhse_scale_1892,feature_vhse_scale_1893,feature_vhse_scale_1894,feature_vhse_scale_1895,feature_vhse_scale_1896
0,-0.67,-0.86,-1.07,-0.41,-0.32,0.27,-0.64,0.11,-1.17,0.7,...,-0.96,-0.52,-1.17,0.7,0.7,0.8,1.64,0.67,1.63,0.13
1,-0.67,-0.86,-1.07,-0.41,-0.32,0.27,-0.64,0.11,-1.17,0.7,...,-0.96,-0.52,-1.17,0.7,0.7,0.8,1.64,0.67,1.63,0.13
2,-0.67,-0.86,-1.07,-0.41,-0.32,0.27,-0.64,0.11,-1.17,0.7,...,-0.96,-0.52,-1.17,0.7,0.7,0.8,1.64,0.67,1.63,0.13
3,-0.67,-0.86,-1.07,-0.41,-0.32,0.27,-0.64,0.11,-1.17,0.7,...,-0.96,-0.52,-1.17,0.7,0.7,0.8,1.64,0.67,1.63,0.13
4,-0.67,-0.86,-1.07,-0.41,-0.32,0.27,-0.64,0.11,-1.17,0.7,...,-0.96,-0.52,-1.17,0.7,0.7,0.8,1.64,0.67,1.63,0.13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31024,-0.67,-0.86,-1.07,-0.41,-0.32,0.27,-0.64,0.11,-1.17,0.7,...,-0.96,-0.52,-1.17,0.7,0.7,0.8,1.64,0.67,1.63,0.13
31025,-0.67,-0.86,-1.07,-0.41,-0.32,0.27,-0.64,0.11,-1.17,0.7,...,-1.61,-0.19,-1.17,0.7,0.7,0.8,1.64,0.67,1.63,0.13
31026,-0.67,-0.86,-1.07,-0.41,-0.32,0.27,-0.64,0.11,-1.17,0.7,...,-0.96,-0.52,-1.17,0.7,0.7,0.8,1.64,0.67,1.63,0.13
31027,-0.67,-0.86,-1.07,-0.41,-0.32,0.27,-0.64,0.11,-1.17,0.7,...,-0.96,-0.52,-1.17,0.7,0.7,0.8,1.64,0.67,1.63,0.13


In [50]:
vhse_columns.to_csv('vhse-scale.csv', index=None)

### Feature Mapping 7 - Z-Scale

In [51]:
zscale = pd.read_csv('Dataset/descriptors/Z-scale.csv', skiprows=[0,1])
zscale_mapping = {}

for index, row in zscale.iterrows():
    zscale_mapping[row['AA_1']] = [row['Z(1)'], row['Z(2)'], row['Z(3)']]

def map_zscale(seq):
    res = []
    for aa in seq:
        res.append(zscale_mapping[aa])
    return np.array(res).flatten().tolist()

In [52]:
zscale_mapping

{'A': [0.07, -1.73, 0.09],
 'R': [2.88, 2.52, -3.44],
 'N': [3.22, 1.45, 0.84],
 'D': [3.64, 1.13, 2.36],
 'C': [0.71, -0.97, 4.13],
 'Q': [2.18, 0.53, -1.14],
 'E': [3.08, 0.39, -0.07],
 'G': [2.23, -5.36, 0.3],
 'H': [2.41, 1.74, 1.11],
 'I': [-4.44, -1.68, -1.03],
 'L': [-4.19, -1.03, -0.98],
 'K': [2.84, 1.41, -3.14],
 'M': [-2.49, -0.27, -0.41],
 'F': [-4.92, 1.3, 0.45],
 'P': [-1.22, 0.88, 2.23],
 'S': [1.96, -1.63, 0.57],
 'T': [0.92, -2.09, -1.4],
 'W': [-4.75, 3.65, 0.85],
 'Y': [-1.39, 2.32, 0.01],
 'V': [-2.69, -2.53, -1.29]}

In [53]:
len(map_zscale("SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLSYGAQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYDYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"))

711

In [54]:
z_columns = pd.DataFrame(train_data['ConstructedAASeq_cln'].apply(map_zscale).tolist(), columns=[f'feature_z_scale_{i+1}' for i in range(711)])

In [55]:
z_columns

,feature_z_scale_1,feature_z_scale_2,feature_z_scale_3,feature_z_scale_4,feature_z_scale_5,feature_z_scale_6,feature_z_scale_7,feature_z_scale_8,feature_z_scale_9,feature_z_scale_10,...,feature_z_scale_702,feature_z_scale_703,feature_z_scale_704,feature_z_scale_705,feature_z_scale_706,feature_z_scale_707,feature_z_scale_708,feature_z_scale_709,feature_z_scale_710,feature_z_scale_711
0,1.96,-1.63,0.57,2.84,1.41,-3.14,2.23,-5.36,0.3,3.08,...,-0.07,-4.19,-1.03,-0.98,-1.39,2.32,0.01,2.84,1.41,-3.14
1,1.96,-1.63,0.57,2.84,1.41,-3.14,2.23,-5.36,0.3,3.08,...,-0.07,-4.19,-1.03,-0.98,-1.39,2.32,0.01,2.84,1.41,-3.14
2,1.96,-1.63,0.57,2.84,1.41,-3.14,2.23,-5.36,0.3,3.08,...,-0.07,-4.19,-1.03,-0.98,-1.39,2.32,0.01,2.84,1.41,-3.14
3,1.96,-1.63,0.57,2.84,1.41,-3.14,2.23,-5.36,0.3,3.08,...,-0.07,-4.19,-1.03,-0.98,-1.39,2.32,0.01,2.84,1.41,-3.14
4,1.96,-1.63,0.57,2.84,1.41,-3.14,2.23,-5.36,0.3,3.08,...,-0.07,-4.19,-1.03,-0.98,-1.39,2.32,0.01,2.84,1.41,-3.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31024,1.96,-1.63,0.57,2.84,1.41,-3.14,2.23,-5.36,0.3,3.08,...,-0.07,2.18,0.53,-1.14,-1.39,2.32,0.01,2.84,1.41,-3.14
31025,1.96,-1.63,0.57,2.84,1.41,-3.14,2.23,-5.36,0.3,3.08,...,-0.07,-4.19,-1.03,-0.98,0.71,-0.97,4.13,2.84,1.41,-3.14
31026,1.96,-1.63,0.57,2.84,1.41,-3.14,2.23,-5.36,0.3,3.08,...,-0.07,-4.19,-1.03,-0.98,-1.39,2.32,0.01,2.84,1.41,-3.14
31027,1.96,-1.63,0.57,2.84,1.41,-3.14,2.23,-5.36,0.3,3.08,...,-0.07,2.18,0.53,-1.14,-1.39,2.32,0.01,2.84,1.41,-3.14


In [56]:
z_columns.to_csv('z_scale.csv', index=None)

## Mapping the descriptors with original training feature set

In [57]:
train_data['DPPS'] = train_data['ConstructedAASeq_cln'].apply(map_dpps)
train_data['MS-WHIM'] = train_data['ConstructedAASeq_cln'].apply(map_mswhim)
train_data['Physical'] = train_data['ConstructedAASeq_cln'].apply(map_physical)
train_data['ST-Scale'] = train_data['ConstructedAASeq_cln'].apply(map_stscale)
train_data['T-Scale'] = train_data['ConstructedAASeq_cln'].apply(map_tscale)
train_data['VHSE-Scale'] = train_data['ConstructedAASeq_cln'].apply(map_vhsescale)
train_data['Z-Scale'] = train_data['ConstructedAASeq_cln'].apply(map_zscale)

In [58]:
train_data.head()

,ConstructedAASeq_cln,Id,Brightness_Class,DPPS,MS-WHIM,Physical,ST-Scale,T-Scale,VHSE-Scale,Z-Scale
0,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,11328,0,"[-1.76, -0.19, 1.06, -0.69, -5.72, 0.14, -4.14...","[-0.8, 0.61, -1.0, -0.51, 0.08, 0.6, -0.31, -0...","[-2.36, 0.38, 0.92, 1.23, -4.04, 0.01, 0.16, 1...","[-1.343, -0.311, -0.917, -0.049, 0.549, -1.533...","[-7.44, -0.65, 0.68, -0.17, 1.58, -2.59, 2.34,...","[-0.67, -0.86, -1.07, -0.41, -0.32, 0.27, -0.6...","[1.96, -1.63, 0.57, 2.84, 1.41, -3.14, 2.23, -..."
1,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,5781,0,"[-1.76, -0.19, 1.06, -0.69, -5.72, 0.14, -4.14...","[-0.8, 0.61, -1.0, -0.51, 0.08, 0.6, -0.31, -0...","[-2.36, 0.38, 0.92, 1.23, -4.04, 0.01, 0.16, 1...","[-1.343, -0.311, -0.917, -0.049, 0.549, -1.533...","[-7.44, -0.65, 0.68, -0.17, 1.58, -2.59, 2.34,...","[-0.67, -0.86, -1.07, -0.41, -0.32, 0.27, -0.6...","[1.96, -1.63, 0.57, 2.84, 1.41, -3.14, 2.23, -..."
2,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,13681,0,"[-1.76, -0.19, 1.06, -0.69, -5.72, 0.14, -4.14...","[-0.8, 0.61, -1.0, -0.51, 0.08, 0.6, -0.31, -0...","[-2.36, 0.38, 0.92, 1.23, -4.04, 0.01, 0.16, 1...","[-1.343, -0.311, -0.917, -0.049, 0.549, -1.533...","[-7.44, -0.65, 0.68, -0.17, 1.58, -2.59, 2.34,...","[-0.67, -0.86, -1.07, -0.41, -0.32, 0.27, -0.6...","[1.96, -1.63, 0.57, 2.84, 1.41, -3.14, 2.23, -..."
3,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,30804,0,"[-1.76, -0.19, 1.06, -0.69, -5.72, 0.14, -4.14...","[-0.8, 0.61, -1.0, -0.51, 0.08, 0.6, -0.31, -0...","[-2.36, 0.38, 0.92, 1.23, -4.04, 0.01, 0.16, 1...","[-1.343, -0.311, -0.917, -0.049, 0.549, -1.533...","[-7.44, -0.65, 0.68, -0.17, 1.58, -2.59, 2.34,...","[-0.67, -0.86, -1.07, -0.41, -0.32, 0.27, -0.6...","[1.96, -1.63, 0.57, 2.84, 1.41, -3.14, 2.23, -..."
4,SKGEELFTGVVPILVELDGDVNGHTFSVSGEGEGDATYGELTLKFI...,30813,0,"[-1.76, -0.19, 1.06, -0.69, -5.72, 0.14, -4.14...","[-0.8, 0.61, -1.0, -0.51, 0.08, 0.6, -0.31, -0...","[-2.36, 0.38, 0.92, 1.23, -4.04, 0.01, 0.16, 1...","[-1.343, -0.311, -0.917, -0.049, 0.549, -1.533...","[-7.44, -0.65, 0.68, -0.17, 1.58, -2.59, 2.34,...","[-0.67, -0.86, -1.07, -0.41, -0.32, 0.27, -0.6...","[1.96, -1.63, 0.57, 2.84, 1.41, -3.14, 2.23, -..."


## Loading the target data - Brightness_Class

In [59]:
data_y = train_data['Brightness_Class']
data_y

0        0
1        0
2        0
3        0
4        0
        ..
31024    0
31025    0
31026    0
31027    1
31028    1
Name: Brightness_Class, Length: 31029, dtype: int64

In [60]:
brightness = train_data['Brightness_Class']

## Data Checkpoint - Feature data Processing is completed

In [61]:
dpps = pd.read_csv('dpps.csv')

ms_whim = pd.read_csv('ms-whim.csv')

physical = pd.read_csv('physical.csv')

st_scale = pd.read_csv('st-scale.csv')

t_scale = pd.read_csv('t-scale.csv')

vhse_scale = pd.read_csv('vhse-scale.csv')

z_scale = pd.read_csv('z_scale.csv')


## Analysing each descriptor/feature separately by running only individual cell for the selected feature to be analysed

In [62]:
X = dpps
y = brightness

In [ ]:
X = ms_whim
y = brightness

In [ ]:
X = physical
y = brightness

In [ ]:
X = st_scale
y = brightness

In [ ]:
X = t_scale
y = brightness

In [ ]:
X = vhse_scale
y = brightness

In [ ]:
X = z_scale
y = brightness

## Analysing all features/descriptors together

In [ ]:
X = pd.concat([dpps, ms_whim, physical, st_scale, t_scale, vhse_scale, z_scale], axis=1)
y = brightness

## Splitting the data into train and test set for Machine Learning

In [63]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=102, stratify=y)

## Analysing results with scaling the data for each feature separately

In [64]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Testing the data on various Models

## 1.  Logistic Regression

In [65]:
# Initialize the model
logistic_model = LogisticRegression(random_state=102, max_iter=10000)

# Train the model
logistic_model.fit(X_train, y_train)

# Predict on test data
logistic_pred = logistic_model.predict(X_test)

# Evaluate
print("Logistic Regression Accuracy:", accuracy_score(y_test, logistic_pred))
print("\nClassification Report:\n", classification_report(y_test, logistic_pred))

Logistic Regression Accuracy: 0.8941347083467612

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.90      0.91      3790
           1       0.85      0.88      0.87      2416

    accuracy                           0.89      6206
   macro avg       0.89      0.89      0.89      6206
weighted avg       0.90      0.89      0.89      6206



## 2.  Random Forest

In [68]:
# Initialize the model
rf_model = RandomForestClassifier(random_state=102, n_estimators=500, n_jobs=-1)

# Train the model
rf_model.fit(X_train, y_train)

# Predict on test data
rf_pred = rf_model.predict(X_test)

# Evaluate
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("\nClassification Report:\n", classification_report(y_test, rf_pred))

Random Forest Accuracy: 0.8491782146310023

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.83      0.87      3790
           1       0.77      0.88      0.82      2416

    accuracy                           0.85      6206
   macro avg       0.84      0.85      0.84      6206
weighted avg       0.86      0.85      0.85      6206



## 3.  SVM

In [69]:
# Initialize the model
svm_model = SVC(kernel='rbf', random_state=102)

# Train the model
svm_model.fit(X_train, y_train)

# Predict on test data
svm_pred = svm_model.predict(X_test)

# Evaluate
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))
print("\nClassification Report:\n", classification_report(y_test, svm_pred))

SVM Accuracy: 0.8696422816629069

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.86      0.89      3790
           1       0.80      0.88      0.84      2416

    accuracy                           0.87      6206
   macro avg       0.86      0.87      0.87      6206
weighted avg       0.87      0.87      0.87      6206



## 4.  MLPClassifier

In [77]:
# Initialize the model
mlp_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=100, random_state=102)

# Train the model
mlp_model.fit(X_train, y_train)

# Predict on test data
mlp_pred = mlp_model.predict(X_test)

# Evaluate
print("MLP Classifier Accuracy:", accuracy_score(y_test, mlp_pred))
print("\nClassification Report:\n", classification_report(y_test, mlp_pred))

MLP Classifier Accuracy: 0.7992265549468257

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.81      0.83      5685
           1       0.72      0.78      0.75      3624

    accuracy                           0.80      9309
   macro avg       0.79      0.80      0.79      9309
weighted avg       0.80      0.80      0.80      9309



### From the above analysis, it is noted that **DPPS descriptor** on a **Logistic Regression Model** with **Scaled data** is able to perform classification with higher F1_score compared to combinations of all features (together and separate) with all models

## Performing feature engineering - analysing if performance increases with a combination of features having good individual F1_scores with DPPS on Logistic Regression Model with Hyperparameter tuning and Regularization

In [ ]:
X = pd.concat([dpps, physical, t_scale], axis=1)
y = brightness

In [70]:
X = pd.concat([dpps,vhse_scale], axis=1)
y = brightness

In [ ]:
X = pd.concat([dpps,physical], axis=1)
y = brightness

In [ ]:
X = pd.concat([dpps, physical, z_scale], axis=1)
y = brightness

In [ ]:
X = pd.concat([dpps, vhse_scale, z_scale], axis=1)
y = brightness

In [ ]:
X = pd.concat([dpps, t_scale, z_scale], axis=1)
y = brightness

In [71]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=102, stratify=y)

In [72]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Upon testing with various regularization parameters, such as L1, L2 and L1+L2, with the saga solver, L1 regularization gave best results

In [74]:
log_reg = LogisticRegression(penalty='l1', solver='saga', max_iter=1000, random_state=102) # Testing with L1 Regularization
log_reg.fit(X_train, y_train)
y_pred_log = log_reg.predict(X_test)
# Evaluate
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print("\nClassification Report:\n", classification_report(y_test, y_pred_log))

Logistic Regression Accuracy: 0.8980019336126329

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.90      0.92      3790
           1       0.85      0.89      0.87      2416

    accuracy                           0.90      6206
   macro avg       0.89      0.90      0.89      6206
weighted avg       0.90      0.90      0.90      6206



/opt/homebrew/anaconda3/envs/CMU_ML/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


### It can be noted that the combination of the two largest sets of features/descriptors - DPPS and VHSE-scale, together yield a good F1 score upon testing on a Logistic Regression Classification Model with L1 Regularization and Saga Solver with more than 1000 iterations

## Trying to check if a subset of feature using Sliding Window improves the F1 score

In [75]:
def filter_features_by_index(df, feature_prefix, start_index, end_index):
    columns_to_include = [f"{feature_prefix}{i}" for i in range(start_index, end_index + 1)]
    filtered_df = df[columns_to_include]
    return filtered_df

In [76]:
slidingwindow = pd.concat([filter_features_by_index(dpps, "feature_dpps_", 490, 2300), filter_features_by_index(vhse_scale, "feature_vhse_scale_", 392, 1840)], axis=1)
X = slidingwindow
y = brightness
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=101)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
log_reg = LogisticRegression(penalty='l1', solver='saga', max_iter=550, random_state=101, C=0.2)
log_reg.fit(X_train, y_train)
y_pred_log = log_reg.predict(X_test)
# Evaluate
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print("\nClassification Report:\n", classification_report(y_test, y_pred_log))

Logistic Regression Accuracy: 0.8359651949726071

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.82      0.86      5685
           1       0.76      0.85      0.80      3624

    accuracy                           0.84      9309
   macro avg       0.83      0.84      0.83      9309
weighted avg       0.84      0.84      0.84      9309



/opt/homebrew/anaconda3/envs/CMU_ML/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


### It can be observed that upon adding a sliding window for taking a subset of combined features with best F1 score so far, the result drops down and thus is not a good approach

## Utility code for constructing the submission for Kaggle Test_X

In [ ]:
test_kaggle_data = pd.read_csv('Dataset/test_X.csv')
feat_columns_test = pd.DataFrame()
test_dpps_data = pd.DataFrame(test_kaggle_data['ConstructedAASeq_cln'].apply(map_dpps).tolist(), columns=[f'feature_dpps_{i+1}' for i in range(2370)])
test_mswhim_data = pd.DataFrame(test_kaggle_data['ConstructedAASeq_cln'].apply(map_mswhim).tolist(), columns=[f'feature_ms_whim_{i+1}' for i in range(711)])
test_physical_data = pd.DataFrame(test_kaggle_data['ConstructedAASeq_cln'].apply(map_physical).tolist(), columns=[f'feature_physical_{i+1}' for i in range(474)])
test_st_scale_data = pd.DataFrame(test_kaggle_data['ConstructedAASeq_cln'].apply(map_stscale).tolist(), columns=[f'feature_st_scale_{i+1}' for i in range(1896)])
test_t_scale_data = pd.DataFrame(test_kaggle_data['ConstructedAASeq_cln'].apply(map_tscale).tolist(), columns=[f'feature_t_scale_{i+1}' for i in range(1185)])
test_vhse_scale_data = pd.DataFrame(test_kaggle_data['ConstructedAASeq_cln'].apply(map_vhsescale).tolist(), columns=[f'feature_vhse_scale_{i+1}' for i in range(1896)])
test_z_scale_data = pd.DataFrame(test_kaggle_data['ConstructedAASeq_cln'].apply(map_zscale).tolist(), columns=[f'feature_z_scale_{i+1}' for i in range(711)])

In [ ]:
kaggle_x = pd.concat([test_dpps_data, test_vhse_scale_data], axis=1)

In [ ]:
kaggle_x_scaled = scaler.fit_transform(kaggle_x)
logistic_pred_kaggle = log_reg.predict(kaggle_x_scaled)
logistic_pred_kaggle
op_kaggle = pd.DataFrame(test_kaggle_data['Id'])
op_kaggle = pd.concat([op_kaggle, pd.DataFrame(logistic_pred_kaggle, columns=['Brightness_Class'])], axis=1)
op_kaggle
op_kaggle.to_csv('output.csv', index=None)

## Conclusion

It can be concluded that for the Classification of Green Fluorescent Protein where we are required to predict the brightness level binarized for classification between high brightness (class 1) and low brightness (class 0) for a set of mutants of Green Fluorescent Protein, the combination of DPPS and VHSE-scale features/descriptors provide the best classification results with scaled data when the classification is performed using Logistic Regression Model with L1 Regularization and Saga Solver for more than 1000 iterations.